In [ ]:
from dotenv import load_dotenv  # import dotenv to read .env file
import os                        # import os to access environment variables

load_dotenv()                              # read key=value pairs from .env into os.environ
api_key = os.getenv("ANTHROPIC_API_KEY")   # retrieve the API key string (None if missing)
print("API key loaded:", api_key is not None)  # confirm key presence without printing secret

In [ ]:
from anthropic import Anthropic  # import the official Anthropic SDK client class

client = Anthropic(api_key=api_key)  # instantiate the client; all API calls go through this object
print("Anthropic client ready")       # confirm the client was created without exception

In [ ]:
response = client.messages.create(        # call the Messages API endpoint
    model="claude-haiku-4-5",             # use Haiku for fast, low-cost smoke test
    #    model="claude-sonnet-4-5",
    max_tokens=50,                        # cap tokens so the smoke test stays cheap
    messages=[{"role": "user", "content": "Say hello in one short sentence."}]  # minimal valid request
)
print(response.content[0].text)  # content is a list of blocks; [0] is the first (text) block

# CCA Lab: Structured Data via Assistant Prefilling and Stop Sequences

**Course:** Building with the Claude API  
**Sub-module:** Accessing Claude with the API  
**Lesson:** Structured Data

## What this lab covers
- Why Claude adds prose wrappers around structured output by default
- Assistant message prefilling to steer output format
- Stop sequences to terminate generation at a predictable boundary
- Combining both techniques to extract clean JSON, code, and lists
- Rubric-driven improvement cycle to measure output quality
- Failure mode analysis: what breaks structured extraction
- Token usage tracking across all API calls in the session

## CCA Domains Exercised
- **Primary:** Prompt Engineering
- **Contributing:** Agentic Architecture (error handling)

## Learning Objectives
1. Explain assistant message prefilling and why it biases Claude's continuation.
2. Explain stop sequences and why they act as hard output terminators.
3. Combine prefilling + stop sequences to extract clean structured data.
4. Validate extracted JSON against a Python schema.
5. Identify at least three failure modes and their symptoms.
6. Track cumulative token usage across a multi-call session.

## Section 1: Prerequisites, Environment Setup, and API Glossary
### CCA objective demonstrated: Identify API parameters and their roles before writing any production code

| Parameter | Type | Required | Purpose |
|-----------|------|----------|---------|
| `model` | string | ✅ | Selects the Claude model (haiku/sonnet/opus). Governs capability and cost. |
| `max_tokens` | int | ✅ | Hard ceiling on output tokens. Prevents runaway costs. |
| `messages` | list[dict] | ✅ | Alternating `user`/`assistant` turns. Last entry drives the next completion. |
| `system` | string | ❌ | Persistent instruction context applied before the conversation. |
| `stop_sequences` | list[str] | ❌ | Strings that terminate generation immediately when encountered. |
| `temperature` | float 0-1 | ❌ | Randomness. 0 = deterministic, 1 = creative. |
| `response.content` | list | — | List of content blocks returned by the API. Usually one TextBlock. |
| `response.stop_reason` | string | — | Why generation stopped: `end_turn`, `stop_sequence`, `max_tokens`. |
| `response.usage` | object | — | `input_tokens` + `output_tokens` for cost tracking. |

**Key insight:** When `stop_reason == "stop_sequence"`, the stop string itself is *not* included in the response text — Claude stopped just before it.

## Section 2: The `ask()` Helper and Usage Logger
### CCA objective demonstrated: Build a reusable API wrapper with full parameter control and telemetry

In [ ]:
usage_log = []  # session-level list; every API call appends one dict for telemetry

def ask(
    messages,            # the full conversation list (user + assistant turns)
    system="",           # optional system prompt; defaults to empty string
    max_tokens=512,      # generous default; callers can override for large outputs
    stop_sequences=None, # None means no stop sequences; pass a list to activate
    label="call"         # human-readable label stored in usage_log for reporting
):
    """
    Thin wrapper around client.messages.create.
    Returns the assistant's text and logs token usage globally.

    Parameters
    ----------
    messages : list[dict]   Full messages list with role/content pairs.
    system   : str          System prompt string (keyword arg per SDK convention).
    max_tokens : int        Maximum tokens the model may generate.
    stop_sequences : list   Strings that terminate generation when encountered.
    label    : str          Tag stored in usage_log for per-call reporting.

    Returns
    -------
    str  The assistant reply text.
    """
    kwargs = {                     # build keyword arguments dict dynamically
        "model": "claude-haiku-4-5",  # Haiku: fastest, cheapest — ideal for labs
        "max_tokens": max_tokens,     # pass caller-supplied ceiling
        "messages": messages,         # full conversation history
    }
    if system:                         # only include system key when non-empty
        kwargs["system"] = system      # system is a top-level key, NOT a message role
    if stop_sequences:                 # only include stop_sequences when caller provides them
        kwargs["stop_sequences"] = stop_sequences  # list of exact strings that halt output

    response = client.messages.create(**kwargs)  # unpack kwargs dict into named parameters

    # response.content is a list of content blocks (TextBlock, ToolUseBlock, etc.)
    # We access [0] because a standard text completion always has exactly one block at index 0
    text = response.content[0].text  # extract the string payload from the first block

    # response.stop_reason tells us WHY generation stopped:
    #   'end_turn'       — Claude finished naturally
    #   'stop_sequence'  — a stop string was hit (the string is NOT in the text)
    #   'max_tokens'     — output was truncated at our ceiling
    stop_reason = response.stop_reason  # capture for logging and debugging

    usage_log.append({               # record telemetry for the token-usage section
        "label": label,              # call identifier (e.g. "baseline", "prefilled")
        "input_tokens": response.usage.input_tokens,   # tokens consumed by the prompt
        "output_tokens": response.usage.output_tokens, # tokens generated in the reply
        "stop_reason": stop_reason,  # useful to verify stop_sequence fired correctly
    })

    return text  # caller receives only the text string; telemetry stays in usage_log


def add_user_message(messages, content):
    """Append a user turn to the messages list in-place."""
    messages.append({"role": "user", "content": content})  # role must be 'user'


def add_assistant_message(messages, content):
    """
    Append a prefilled assistant turn to the messages list in-place.
    This is the 'assistant prefilling' technique: the API sees this text
    as if Claude already wrote it, so Claude's completion continues from here.
    """
    messages.append({"role": "assistant", "content": content})  # role must be 'assistant'


print("ask() helper and message utilities defined.")  # confirm no import/syntax errors

## Section 3: Structured Output via Assistant Prefilling and Stop Sequences
### CCA objective demonstrated: Use assistant prefilling + stop_sequences to extract clean structured data without prose wrappers

**The core problem:** Claude defaults to being helpful — it wraps JSON in markdown fences and adds explanatory prose. For applications that parse the raw response (e.g., `json.loads()`), this extra text raises `JSONDecodeError`.

**The two-part solution:**
1. **Assistant prefilling** — add an `assistant` turn containing the opening of a markdown code block (`` ```json ``). Claude sees this as text it already wrote and continues from that point.
2. **Stop sequence** — pass `stop_sequences=["```"]`. When Claude tries to close the code block, generation halts *before* the closing fence. The result is just the JSON body.

**Sequence of events:**
```
User:      "Generate a short EventBridge rule as JSON"
Assistant: "```json"              ← prefilled (we wrote this)
           { ... JSON ... }       ← Claude writes this
           "```"                  ← STOP — generation ends here, string excluded
```

In [ ]:
# Helper defined separately so f-strings in the comparison table stay clean (Rule: no backslash in f-string)
def _baseline_parseable(text):
    """Return True if text is valid JSON, False otherwise."""
    try:
        json.loads(text)   # attempt parse
        return True        # no exception → parseable
    except Exception:
        return False       # any error → not parseable


In [ ]:

# Re-run the comparison table now that the helper exists
baseline_preview  = baseline_output.replace('\n', ' ')[:60]
prefilled_preview = prefilled_output.replace('\n', ' ')[:60]

baseline_ok  = "✅" if _baseline_parseable(baseline_output) else "❌"
prefilled_ok = "✅" if _baseline_parseable(clean_json_str)  else "❌"

print("=== SIDE-BY-SIDE COMPARISON ===")
print(f"{'Technique':<20} {'Output Preview (60 chars)':<62} {'Parseable?':<12}")
print("-" * 96)
print(f"{'Baseline':<20} {baseline_preview:<62} {baseline_ok:<12}")
print(f"{'Prefill+Stop':<20} {prefilled_preview:<62} {prefilled_ok:<12}")

# Schema validation: check that the parsed JSON has expected EventBridge keys
print()
print("=== SCHEMA VALIDATION ===")
required_keys = ["source", "detail-type"]  # minimum EventBridge rule fields
for key in required_keys:                  # iterate over required key names
    icon = "✅" if key in parsed else "❌"   # check membership in parsed dict
    print(f"  {icon} Field '{key}' present: {key in parsed}")  # print result per field
import json  # standard library: parse JSON strings into Python dicts

# ── BASELINE: no prefilling, no stop sequences ────────────────────────────
baseline_messages = []                          # fresh conversation list
add_user_message(                               # append the user turn
    baseline_messages,
    "Generate a very short AWS EventBridge rule as JSON. "
    "The rule should match EC2 instance state changes to 'running'."
)

# ask() with no stop_sequences — Claude will do whatever it likes
baseline_output = ask(baseline_messages, label="baseline")  # call API, log usage

print("=== BASELINE OUTPUT (raw, no technique applied) ===")
print(baseline_output)          # show the full response including prose and fences
print()

# Attempt to parse baseline as JSON — this will likely fail
try:
    json.loads(baseline_output)              # try direct parse
    print("✅ Baseline parsed as JSON (Claude was unusually clean)")
except json.JSONDecodeError as e:
    print(f"❌ Baseline JSONDecodeError: {e}")  # expected failure: fences + prose break parsing

print()

# ── PREFILLED: assistant prefilling + stop sequence ───────────────────────
prefilled_messages = []                         # separate conversation list
add_user_message(                               # same user request
    prefilled_messages,
    "Generate a very short AWS EventBridge rule as JSON. "
    "The rule should match EC2 instance state changes to 'running'."
)
add_assistant_message(prefilled_messages, "```json")  # prefill: Claude thinks it started the fence

# stop_sequences=["```"] halts generation the instant Claude tries to close the fence
# The closing ``` is NOT included in the returned text
prefilled_output = ask(
    prefilled_messages,
    stop_sequences=["```"],  # terminate at the closing code fence
    label="prefilled"        # tag for usage_log
)

print("=== PREFILLED OUTPUT (raw, before .strip()) ===")
print(repr(prefilled_output))   # repr() shows \n characters so we can see whitespace
print()

# Clean up leading/trailing whitespace introduced by the prefill gap
clean_json_str = prefilled_output.strip()  # .strip() removes leading/trailing \n and spaces

try:
    parsed = json.loads(clean_json_str)     # parse the stripped string into a Python dict
    print("✅ Prefilled output parsed successfully as JSON:")
    print(json.dumps(parsed, indent=2))     # pretty-print for readability
except json.JSONDecodeError as e:
    print(f"❌ JSONDecodeError: {e}")
    print("Raw output was:", repr(clean_json_str))

print()

# ── SIDE-BY-SIDE COMPARISON TABLE ────────────────────────────────────────
# Truncate outputs to 60 chars so the table stays readable in narrow terminals
baseline_preview = baseline_output.replace('\n', ' ')[:60]   # collapse newlines, truncate
prefilled_preview = prefilled_output.replace('\n', ' ')[:60] # same for prefilled

print("=== SIDE-BY-SIDE COMPARISON ===")
print(f"{'Technique':<20} {'Output Preview (60 chars)':<62} {'Parseable?':<12}")
print("-" * 96)

# Check parseability for table — evaluate separately so f-string stays clean
baseline_ok = "✅" if _baseline_parseable(baseline_output) else "❌"
prefilled_ok = "✅"
try:
    json.loads(clean_json_str)
except Exception:
    prefilled_ok = "❌"

print(f"{'Baseline':<20} {baseline_preview:<62} {baseline_ok:<12}")
print(f"{'Prefill+Stop':<20} {prefilled_preview:<62} {prefilled_ok:<12}")

## Section 4: Intentional Error Demonstration — What Breaks Structured Output
### CCA objective demonstrated: Identify failure modes in structured extraction and diagnose them from error messages

This section deliberately omits or misconfigures the prefill+stop technique to show what goes wrong and why the parser fails.

In [ ]:
import re  # standard library: regex for inspecting raw output

print("===== FAILURE MODE 1: Prefill present but stop_sequences omitted =====")
# Without stop_sequences, Claude closes the fence itself — the ``` ending up in the text
fail1_messages = []
add_user_message(fail1_messages, "Generate a very short EventBridge rule as JSON.")
add_assistant_message(fail1_messages, "```json")  # prefill is there
# BUG: stop_sequences intentionally omitted
fail1_output = ask(fail1_messages, label="fail1-no-stop")  # Claude appends closing ```
print("Raw output:")
print(fail1_output)
print()
try:
    json.loads(fail1_output.strip())  # will fail because closing ``` is included
    print("✅ Parsed (unexpected)")
except json.JSONDecodeError as e:
    print(f"❌ JSONDecodeError: {e}")
    print("  → Diagnosis: closing ``` fence is still in the text because stop_sequences was omitted.")

print()
print("===== FAILURE MODE 2: No prefill, no stop_sequences =====")
# Default behaviour: Claude adds prose, markdown, explanations
fail2_messages = []
add_user_message(fail2_messages, "Generate a very short EventBridge rule as JSON.")
# BUG: neither prefill nor stop_sequences
fail2_output = ask(fail2_messages, label="fail2-no-technique")  # fully wrapped output
print("Raw output:")
print(fail2_output)
print()
try:
    json.loads(fail2_output.strip())
    print("✅ Parsed (Claude was unusually clean)")
except json.JSONDecodeError as e:
    print(f"❌ JSONDecodeError: {e}")
    print("  → Diagnosis: prose and/or markdown fences contaminate the JSON string.")

print()
print("===== FAILURE MODE 3: Wrong stop sequence string =====")
# Using '```python' instead of '```' — the stop never fires for a JSON block
fail3_messages = []
add_user_message(fail3_messages, "Generate a very short EventBridge rule as JSON.")
add_assistant_message(fail3_messages, "```json")
# BUG: wrong stop sequence — will never match the ``` that closes the JSON block
fail3_output = ask(fail3_messages, stop_sequences=["```python"], label="fail3-wrong-stop")
print("Raw output:")
print(fail3_output)
print()
try:
    json.loads(fail3_output.strip())
    print("✅ Parsed (unexpected)")
except json.JSONDecodeError as e:
    print(f"❌ JSONDecodeError: {e}")
    print("  → Diagnosis: stop sequence '```python' never matched the closing '```', fence leaks in.")

## Section 5: Improvement Cycle — Write → Evaluate → Improve → Re-evaluate
### CCA objective demonstrated: Apply a numeric rubric to prompt iterations and measure score improvement

We test three prompt versions against a 4-dimension rubric for generating a clean, structured product description:

| Dimension | What we check | Points |
|-----------|--------------|--------|
| **Action verb start** | First prose sentence starts with an action verb (no markdown headings) | 1 |
| **Keyword 'features'** | Response contains the word 'features' (case-insensitive) | 1 |
| **Under 80 words** | Total word count ≤ 80 | 1 |
| **No markdown fences** | No triple-backtick fences in the output | 1 |

**Threshold: 4/4** (only V3 should cross it)

> **Grader reliability note:** Deterministic rubrics can over-score responses that
> contain the right words but wrong semantics. For production evals, complement
> keyword checks with a Claude-as-judge semantic pass — the rule + judge layering
> pattern from the evaluation labs.

In [ ]:
import re  # already imported above; re-import is harmless and makes cell self-contained

PASS_THRESHOLD = 4  # V3 must reach this score; V1 and V2 must stay below it

def score_product_description(text):
    """
    Score a product description on 4 rubric dimensions.

    Parameters
    ----------
    text : str  The assistant's response text.

    Returns
    -------
    int  Total score 0-4.
    """
    total = 0  # accumulator for dimension scores

    # ── Dimension 1: First prose sentence starts with an action verb ────────
    # Rule A: skip markdown heading lines (lines starting with #) to find real first sentence
    first_sentence = next(
        (s.strip() for s in text.split('\n')     # iterate lines
         if s.strip() and not s.strip().startswith('#')),  # skip blank + heading lines
        ''  # default if all lines are headings
    )
    # re.match checks from the very start of the string — correct for 'starts with'
    # \b word boundary ensures we match whole words only
    action_verbs = r'\b(Discover|Explore|Boost|Transform|Elevate|Get|Build|Create|Unlock|Use)\b'
    d1 = int(bool(re.match(action_verbs, first_sentence, re.IGNORECASE)))
    # int(bool(...)) converts regex Match object (truthy) or None (falsy) to 0 or 1
    icon1 = "✅" if d1 else "❌"
    print(f"  {icon1} D1 Action verb start ({first_sentence[:40]!r}): +{d1}")
    total += d1  # add dimension score to total

    # ── Dimension 2: Response contains the word 'features' ──────────────────
    # re.search (not re.match) because 'features' can appear anywhere in the text
    # \b word boundaries prevent matching 'featured', 'featureless', etc.
    d2 = int(bool(re.search(r'\bfeatures\b', text, re.IGNORECASE)))
    icon2 = "✅" if d2 else "❌"
    print(f"  {icon2} D2 Contains 'features': +{d2}")
    total += d2

    # ── Dimension 3: Total word count under 80 ──────────────────────────────
    # text.split() splits on any whitespace; len() counts the resulting tokens
    word_count = len(text.split())
    d3 = int(word_count <= 80)  # int(bool()) pattern: True→1, False→0
    icon3 = "✅" if d3 else "❌"
    print(f"  {icon3} D3 Under 80 words ({word_count} words): +{d3}")
    total += d3

    # ── Dimension 4: No markdown triple-backtick fences ─────────────────────
    # re.search checks anywhere in text; we want NO match, so we invert with 'not'
    d4 = int(not bool(re.search(r'```', text)))
    icon4 = "✅" if d4 else "❌"
    print(f"  {icon4} D4 No markdown fences: +{d4}")
    total += d4

    print(f"  ── Total: {total}/{PASS_THRESHOLD}")
    return total  # return integer score for comparison table

print("score_product_description() defined — 4 dimensions, threshold =", PASS_THRESHOLD)

In [ ]:
# ── V1: Vague prompt — expected to fail D1, D2, D3 ──────────────────────────
# Fails D1 because no instruction to start with an action verb
# Fails D2 because no instruction to include 'features'
# Fails D3 because no word-count constraint (Claude tends to over-explain)
v1_prompt = "Write a product description for wireless noise-cancelling headphones."
v1_msgs = []                              # fresh conversation list for V1
add_user_message(v1_msgs, v1_prompt)      # single user turn
v1_response = ask(v1_msgs, label="v1")    # call API, log usage
print("── V1 Score ──")
v1_score = score_product_description(v1_response)  # print per-dimension + total

print()

# ── V2: Partial fix — adds word count but not verb/features ──────────────────
# Fixes D3 (word count) but still fails D1 (action verb) and D2 (features keyword)
v2_prompt = (
    "Write a product description for wireless noise-cancelling headphones. "
    "Keep it under 80 words and do not use markdown formatting."
)
v2_msgs = []                              # fresh conversation list for V2
add_user_message(v2_msgs, v2_prompt)
v2_response = ask(v2_msgs, label="v2")
print("── V2 Score ──")
v2_score = score_product_description(v2_response)

print()

# ── V3: Full system prompt covering all failing dimensions ────────────────────
# Fixes D1: 'Start your response with an action verb'
# Fixes D2: 'include the word features'
# Fixes D3: 'under 80 words'
# Fixes D4: 'Do not use markdown headings or code fences'
v3_system = (
    "You write concise, punchy product copy. "
    "Do not use markdown headings or code fences. "
    "Start your response with an action verb. "
    "Always include the word 'features' in your response. "
    "Keep responses under 80 words."
)
v3_prompt = "Write a product description for wireless noise-cancelling headphones."
v3_msgs = []                              # fresh conversation list for V3
add_user_message(v3_msgs, v3_prompt)
v3_response = ask(v3_msgs, system=v3_system, label="v3")  # system prompt applied
print("── V3 Score ──")
v3_score = score_product_description(v3_response)

print()

# ── SIDE-BY-SIDE COMPARISON TABLE ────────────────────────────────────────────
# Truncate prompt and response previews to keep table columns aligned
def _preview(s, n=45):
    """Return first n chars of s with newlines collapsed."""
    return s.replace('\n', ' ')[:n]  # collapse newlines, then truncate

print("=" * 80)
print("IMPROVEMENT CYCLE SUMMARY")
print("=" * 80)
header = f"{'Ver':<5} {'Prompt (45 chars)':<47} {'Response (45 chars)':<47} {'Score':<6}"
print(header)
print("-" * 80)
# Build rows as variables — avoids backslash in f-string expressions
v1_row = f"{'V1':<5} {_preview(v1_prompt):<47} {_preview(v1_response):<47} {v1_score}/{PASS_THRESHOLD}"
v2_row = f"{'V2':<5} {_preview(v2_prompt):<47} {_preview(v2_response):<47} {v2_score}/{PASS_THRESHOLD}"
v3_row = f"{'V3':<5} {_preview(v3_prompt):<47} {_preview(v3_response):<47} {v3_score}/{PASS_THRESHOLD}"
print(v1_row)
print(v2_row)
print(v3_row)
print("=" * 80)

# Conditional PASS/FAIL line based on threshold
v3_verdict = "PASS" if v3_score >= PASS_THRESHOLD else "FAIL"
v2_verdict = "FAIL" if v2_score < PASS_THRESHOLD else "PASS (unexpected)"
v1_verdict = "FAIL" if v1_score < PASS_THRESHOLD else "PASS (unexpected)"
print(f"V1: {v1_verdict}  |  V2: {v2_verdict}  |  V3: {v3_verdict}  (threshold={PASS_THRESHOLD})")

## Section 6: Failure Mode Analysis
### CCA objective demonstrated: Catalogue structured-output failure modes and reproduce them with live code

| Failure Mode | Trigger | Symptom |
|---|---|---|
| **No stop sequence** | `stop_sequences` omitted while using prefill | Closing ` ``` ` appears in text; `json.loads()` raises `JSONDecodeError` |
| **Wrong stop sequence** | Stop string doesn't match the actual closing delimiter | Stop never fires; full markdown block (with fence) is returned |
| **No prefill, no stop** | Plain user-only message asking for JSON | Claude adds prose, headers, and fences; raw output is not parseable |
| **max_tokens too low** | Ceiling hit mid-JSON object | Truncated JSON with no closing `}` or `]`; parse error or silent data loss |
| **Prefill mismatch** | Prefill says ` ```python ` but content is JSON | Claude may switch format or add mixed content to close the 'wrong' block |

In [ ]:
print("===== LIVE FAILURE: max_tokens truncation mid-JSON =====")
# Deliberately set max_tokens=30 — far too small for a complete JSON object
trunc_messages = []
add_user_message(trunc_messages, "Generate a detailed AWS EventBridge rule as JSON with multiple fields.")
add_assistant_message(trunc_messages, "```json")  # prefill the fence

# Call API directly (not via ask()) so we can inspect stop_reason without
# the helper hiding it; we still log usage manually
trunc_response = client.messages.create(
    model="claude-haiku-4-5",   # Haiku is fast for live demos
    max_tokens=30,              # intentionally tiny — will truncate mid-JSON
    messages=trunc_messages,    # includes the prefilled assistant turn
    stop_sequences=["```"],     # stop sequence is correct, but truncation fires first
)

# Log this call manually since we bypassed ask()
usage_log.append({
    "label": "fail-truncation",
    "input_tokens": trunc_response.usage.input_tokens,
    "output_tokens": trunc_response.usage.output_tokens,
    "stop_reason": trunc_response.stop_reason,
})

trunc_text = trunc_response.content[0].text   # extract text from first content block
print(f"stop_reason: {trunc_response.stop_reason}")  # expect 'max_tokens'
print(f"Raw output: {repr(trunc_text)}")
print()

try:
    json.loads(trunc_text.strip())  # truncated JSON won't parse
    print("✅ Parsed (unexpectedly complete despite truncation)")
except json.JSONDecodeError as e:
    print(f"❌ JSONDecodeError: {e}")
    print("  → Diagnosis: max_tokens={} cut the output before closing brace.".format(30))
    print("  → Fix: increase max_tokens or instruct Claude to keep JSON compact.")

print()
print("===== LIVE FAILURE: Prefill/stop mismatch (prefill says ```python, content is JSON) =====")
mismatch_messages = []
add_user_message(mismatch_messages, "Generate a short AWS EventBridge rule as JSON.")
add_assistant_message(mismatch_messages, "```python")  # WRONG: should be ```json
mismatch_output = ask(
    mismatch_messages,
    stop_sequences=["```"],   # stop sequence is correct, but prefill is misleading
    label="fail-mismatch"
)
print("Raw output:")
print(mismatch_output)
try:
    json.loads(mismatch_output.strip())
    print("✅ Parsed (Claude ignored the python hint and wrote JSON anyway)")
except json.JSONDecodeError as e:
    print(f"❌ JSONDecodeError: {e}")
    print("  → Diagnosis: ```python hint caused Claude to switch to Python dict syntax.")

## Section 7: Token Usage Tracking Across the Session
### CCA objective demonstrated: Monitor per-call and cumulative token consumption to estimate cost and detect inefficiencies

In [ ]:
# Print a per-call usage table with cumulative running totals
print("=" * 72)
print("TOKEN USAGE LOG — all API calls in this session")
print("=" * 72)
header = f"{'#':<4} {'Label':<24} {'Input':>8} {'Output':>8} {'CumIn':>8} {'CumOut':>8} {'Stop Reason':<16}"
print(header)
print("-" * 72)

cum_input  = 0   # running total of input tokens across all calls
cum_output = 0   # running total of output tokens across all calls

for i, entry in enumerate(usage_log, start=1):  # enumerate starting at 1 for human-friendly numbering
    cum_input  += entry["input_tokens"]   # add this call's input to running total
    cum_output += entry["output_tokens"]  # add this call's output to running total

    # Build row as a variable — avoids backslash inside f-string expression
    row = (
        f"{i:<4} "
        f"{entry['label']:<24} "      # label stored when ask() was called
        f"{entry['input_tokens']:>8} "  # input tokens for this call
        f"{entry['output_tokens']:>8} " # output tokens for this call
        f"{cum_input:>8} "              # cumulative input so far
        f"{cum_output:>8} "             # cumulative output so far
        f"{entry['stop_reason']:<16}"   # why generation stopped
    )
    print(row)

print("-" * 72)
# Summary line
total_tokens = cum_input + cum_output  # grand total tokens this session
print(f"{'TOTAL':<30} {cum_input:>8} {cum_output:>8}  (grand total: {total_tokens})")
print("=" * 72)
print()
print(f"Calls logged: {len(usage_log)}")
print(f"Estimated cost (Haiku ~$0.25/M input, $1.25/M output):")
# Calculate cost estimate separately to keep f-string readable
input_cost  = cum_input  / 1_000_000 * 0.25  # dollars
output_cost = cum_output / 1_000_000 * 1.25  # dollars
total_cost  = input_cost + output_cost
print(f"  Input:  ${input_cost:.6f}")
print(f"  Output: ${output_cost:.6f}")
print(f"  Total:  ${total_cost:.6f}")

## Section 8: Multi-Turn Conversation — Messages List Accumulation
### CCA objective demonstrated: Build a multi-turn conversation and observe the messages list grow after each turn

The messages list is the API's memory. Every call to `client.messages.create()` is stateless — we must pass the full conversation history each time. This section shows the list after each turn so you can see it accumulate.

In [ ]:
def print_messages(messages, turn_label):
    """
    Print every entry in the messages list with role and content preview.
    Called after each turn so the reader sees the list grow.
    """
    print(f"\n── Messages list after {turn_label} ({len(messages)} entries) ──")
    for idx, msg in enumerate(messages):        # iterate with index for display
        role    = msg["role"]                   # 'user' or 'assistant'
        content = msg["content"]                # string content of this turn
        preview = content.replace('\n', ' ')[:80]  # collapse newlines, limit width
        print(f"  [{idx}] {role:<12} | {preview}")  # show index, role, preview


# Start a multi-turn conversation about structured data techniques
conv = []  # empty messages list — conversation starts here

# ── Turn 1 ──────────────────────────────────────────────────────────────
add_user_message(conv, "What is assistant message prefilling and why does it help with JSON extraction?")
print_messages(conv, "Turn 1 (user)")  # print list after adding user turn

reply1 = ask(conv, max_tokens=150, label="multi-turn-1")  # get Claude's answer
add_assistant_message(conv, reply1)    # append Claude's reply to keep history
print_messages(conv, "Turn 1 (assistant)")  # print list — now has 2 entries

# ── Turn 2 ──────────────────────────────────────────────────────────────
add_user_message(conv, "And what role do stop sequences play? Give a one-sentence answer.")
print_messages(conv, "Turn 2 (user)")  # print list — now has 3 entries

reply2 = ask(conv, max_tokens=80, label="multi-turn-2")
add_assistant_message(conv, reply2)
print_messages(conv, "Turn 2 (assistant)")  # print list — now has 4 entries

# ── Turn 3 ──────────────────────────────────────────────────────────────
add_user_message(conv, "Summarise both techniques in exactly 10 words.")
print_messages(conv, "Turn 3 (user)")  # print list — now has 5 entries

reply3 = ask(conv, max_tokens=40, label="multi-turn-3")
add_assistant_message(conv, reply3)
print_messages(conv, "Turn 3 (assistant)")  # print list — now has 6 entries (full history)

print("\nFinal assistant summary:", reply3)

## Section 9: Practice Drills
### CCA objective demonstrated: Apply prefilling and stop sequences independently to new structured-data tasks

In [ ]:
# ── Drill 1: Extract a Python list of strings ────────────────────────────
# Task: Use prefilling + stop sequences to get a bare Python list of 5 fruit names.
# Prefill with '[' and stop at ']' to extract just the list body.
print("=== Drill 1: Python list extraction ===")
d1_msgs = []                                      # fresh conversation
add_user_message(d1_msgs, "Give me a Python list of exactly 5 tropical fruit names.")  # user request
add_assistant_message(d1_msgs, "[")               # prefill the opening bracket
d1_output = ask(d1_msgs, stop_sequences=["]"], label="drill1")  # stop at closing bracket
# Reconstruct the full list string from prefill + output + stop char
full_list_str = "[" + d1_output + "]"             # reassemble: prefill + body + stop char
print("Raw output:", repr(d1_output))
print("Reconstructed:", full_list_str)
try:
    import ast
    parsed_list = ast.literal_eval(full_list_str)  # safe eval for Python literals
    print("✅ Parsed list:", parsed_list)
except Exception as e:
    print(f"❌ Parse error: {e}")

print()

# ── Drill 2: Extract CSV data ────────────────────────────────────────────
# Task: Get raw CSV (no headers prose) for 3 AWS regions with latency values.
print("=== Drill 2: CSV data extraction ===")
d2_msgs = []
add_user_message(d2_msgs, "Generate a 3-row CSV with columns: region,latency_ms for top AWS regions.")
# No code fence for CSV — prefill directly with the header row
add_assistant_message(d2_msgs, "region,latency_ms")   # prefill the CSV header
d2_output = ask(d2_msgs, max_tokens=80, label="drill2")  # no special stop — just get the rows
csv_full = "region,latency_ms" + d2_output           # reattach prefilled header
print("CSV output:")
print(csv_full)
# Validate by counting lines
lines = [ln for ln in csv_full.strip().split('\n') if ln.strip()]  # filter empty lines
print(f"Total lines (header + data): {len(lines)}")

print()

# ── Drill 3: Extract a YAML snippet ─────────────────────────────────────
# Task: Get clean YAML for a Kubernetes ConfigMap without prose wrappers.
print("=== Drill 3: YAML extraction ===")
d3_msgs = []
add_user_message(d3_msgs, "Generate a minimal Kubernetes ConfigMap YAML for app config with one key: LOG_LEVEL: info")
add_assistant_message(d3_msgs, "```yaml")           # prefill the yaml fence
d3_output = ask(d3_msgs, stop_sequences=["```"], label="drill3")  # stop at closing fence
yaml_clean = d3_output.strip()                      # strip leading/trailing whitespace
print("Clean YAML output:")
print(yaml_clean)
# Basic validation: check for 'ConfigMap' and 'LOG_LEVEL'
for keyword in ["ConfigMap", "LOG_LEVEL"]:
    icon = "✅" if keyword in yaml_clean else "❌"
    print(f"  {icon} Contains '{keyword}': {keyword in yaml_clean}")

## Section 10: CCA Takeaways — 5 Exam-Ready Mental Models
### CCA objective demonstrated: Consolidate structured-data concepts into portable, exam-applicable mental models

| # | Mental Model | Exam Scenario Application |
|---|---|---|
| 1 | **Prefill = bias, not guarantee** — Putting ` ```json ` in the assistant turn biases Claude to continue with JSON, but only a matching stop sequence *guarantees* the fence doesn't leak into the output. | Structured Data Extraction: choose the right combination for parse-safe output. |
| 2 | **Stop sequence is a hard boundary** — The matching string is never included in `response.content[0].text`. You must reattach it yourself if the format requires it (e.g., ` ] ` for a list). | Structured Data Extraction: know when to reconstruct the delimiter after extraction. |
| 3 | **`stop_reason` is your diagnostic** — `end_turn` means natural completion; `stop_sequence` means the technique fired correctly; `max_tokens` means truncation — the most dangerous failure mode for JSON. | Agentic Architecture: inspect stop_reason in production to detect silent data loss. |
| 4 | **System prompt = persistent format contract** — For consistent structured output across many turns, encode format rules in the system prompt rather than repeating them in every user message. | Prompt Engineering: use system for invariant constraints, user for variable content. |
| 5 | **Deterministic rubrics + semantic judge = reliable eval** — Keyword checks are fast but can miss semantic failures. Layer a Claude-as-judge pass on top of regex scoring for production eval pipelines. | Evaluation design: know when keyword scoring is sufficient vs. when LLM-as-judge is required. |